# Flood Event Cross-Asset Integration Demo

這份 notebook 示範如何把**同一個淹水事件**中的異質資料整合在一起分析：
- `depth_iot`：IoT 觀測與對應模擬值（Parquet）
- `depth_simulation`：完整模擬格網時序（Zarr）
- `sim_depth_at_stations`：站點模擬時序 fallback（CSV）

你會看到兩個關鍵價值：
1. 用單一 STAC item 管理跨格式資產
2. 透過 `mesh2d_face_index / grid_id` 完成點位-格網對齊


## Step 1. 初始化與通用工具
先定義路徑、STAC href 轉本機路徑、Zarr 安全開啟策略，以及時間轉換函式。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# 專案根目錄與目標 STAC item
REPO = Path('/Users/insight_ai/Documents/wangup-stac-manager')
ITEM_PATH = REPO / 'stac_catalog/flood_modelling/flood_event_20210604/items/flood_event_20210604__integrated.json'
assert ITEM_PATH.exists(), f'Missing item: {ITEM_PATH}'

def stac_href_to_local_path(href: str, repo_root: Path) -> Path:
    """把 STAC `/stac/...` href 轉成工作區的實體路徑。"""
    if href.startswith('/stac/'):
        return repo_root / 'stac_catalog' / href.removeprefix('/stac/')
    return Path(href)

def open_zarr_safe(path: Path):
    """
    優先嘗試 consolidated metadata；
    若環境與 store 不相容，自動 fallback。
    回傳: (dataset_or_none, mode_text)
    """
    try:
        ds = xr.open_zarr(path, consolidated=True)
        return ds, 'zarr(consolidated=True)'
    except Exception as e1:
        try:
            ds = xr.open_zarr(path, consolidated=False)
            return ds, f'zarr(consolidated=False), fallback_from={type(e1).__name__}'
        except Exception as e2:
            return None, f'zarr_unavailable: {type(e1).__name__}: {e1} | {type(e2).__name__}: {e2}'

def decode_time_raw_with_units(raw_values: pd.Series, units: str) -> pd.DatetimeIndex:
    """將 fallback CSV 的 time_raw + time_units 轉成 UTC DatetimeIndex。"""
    unit_map = {
        'second': 's', 'seconds': 's',
        'minute': 'm', 'minutes': 'm',
        'hour': 'h', 'hours': 'h',
        'day': 'D', 'days': 'D',
    }
    if ' since ' not in units:
        # 若格式不符合標準，退回一般 datetime 解析
        return pd.to_datetime(raw_values, utc=True, errors='coerce')

    left, base = units.split(' since ', 1)
    left = left.strip().lower()
    base_ts = pd.to_datetime(base.strip(), utc=True, errors='coerce')
    if pd.isna(base_ts):
        return pd.to_datetime(raw_values, utc=True, errors='coerce')

    td_unit = unit_map.get(left, 's')
    offsets = pd.to_timedelta(pd.to_numeric(raw_values, errors='coerce'), unit=td_unit)
    return pd.DatetimeIndex(base_ts + offsets)


## Step 2. 讀取 integrated item 與資產索引
這步會確認 STAC item 內實際有哪些資產，以及它們對應到本機哪個檔案。

In [ ]:
item = json.loads(ITEM_PATH.read_text(encoding='utf-8'))
asset_paths = {k: stac_href_to_local_path(v['href'], REPO) for k, v in item['assets'].items()}

print('Item:', item['id'])
print('Assets:', sorted(asset_paths.keys()))
asset_paths


## Step 3. 載入 IoT Parquet 與 mapping index
- `depth_iot` 提供觀測值與站點對應欄位
- `station_face_mapping` 說明站點如何映射到 mesh face

In [ ]:
iot_df = pd.read_parquet(asset_paths['depth_iot'])
iot_df['time_utc'] = pd.to_datetime(iot_df['time_utc'], utc=True, errors='coerce')
mapping = json.loads(asset_paths['station_face_mapping'].read_text(encoding='utf-8'))

print('IoT rows:', len(iot_df))
print('Stations:', iot_df['station_id'].nunique())
print('Mapping method:', mapping.get('mapping_method'))
iot_df.head(3)


## Step 4. 定義模擬序列載入函式（Zarr 優先，CSV 備援）
這個函式是跨環境相容的核心：
- 若 Zarr 可讀：從完整格網抽出指定 face 的時序
- 若 Zarr 不可讀：改用 `sim_depth_at_stations` CSV 取得同站點時序

In [ ]:
def load_sim_series_for_station(station_rows: pd.DataFrame, asset_paths: dict):
    # 以站點對應到的 mesh face 作為模擬抽樣索引
    station_id = station_rows['station_id'].dropna().iloc[0]
    face_idx = int(station_rows['mesh2d_face_index'].dropna().iloc[0])

    # 路徑 A: 直接讀 Zarr
    ds, mode = open_zarr_safe(asset_paths['depth_simulation'])
    if ds is not None:
        face_dim = next((d for d in ds['Mesh2d_waterdepth'].dims if 'face' in d.lower()), None)
        if face_dim is not None:
            s = ds['Mesh2d_waterdepth'].isel({face_dim: face_idx}).to_series()
            s.index = pd.to_datetime(s.index, utc=True, errors='coerce')
            return s.rename('sim_depth_model_m'), face_idx, mode

    # 路徑 B: fallback CSV（針對舊 kernel 或 zarr 不相容環境）
    if 'sim_depth_at_stations' not in asset_paths:
        raise RuntimeError(f'Cannot read Zarr and no sim_depth_at_stations asset available. zarr_mode={mode}')

    sim_csv = pd.read_csv(asset_paths['sim_depth_at_stations'])
    sim_csv = sim_csv[sim_csv['station'] == station_id].copy()
    if sim_csv.empty:
        raise RuntimeError(f'Fallback CSV has no station rows for: {station_id}')

    units = str(sim_csv['time_units'].dropna().iloc[0]) if 'time_units' in sim_csv else 'seconds since 1970-01-01 00:00:00 +00:00'
    sim_csv['time_utc'] = decode_time_raw_with_units(sim_csv['time_raw'], units)
    sim_csv['simulated_depth_m'] = pd.to_numeric(sim_csv['simulated_depth_m'], errors='coerce')
    s = sim_csv.set_index('time_utc')['simulated_depth_m'].sort_index()
    return s.rename('sim_depth_model_m'), face_idx, f'fallback_csv ({mode})'


## Step 5. 以單一測站做對齊
這裡先選第一個測站示範：
- 模擬序列：來自 Zarr 或 CSV fallback
- 觀測序列：來自 IoT parquet
- 並行保留 parquet 內的模擬欄位，方便比對資料一致性

In [ ]:
station_id = iot_df['station_id'].dropna().iloc[0]
station_rows = iot_df[iot_df['station_id'] == station_id].copy().sort_values('time_utc')

sim_series, face_idx, sim_mode = load_sim_series_for_station(station_rows, asset_paths)
obs_series = station_rows.set_index('time_utc')['observed_depth_m'].rename('obs_depth_iot_m')
sim_iot_series = station_rows.set_index('time_utc')['simulated_depth_m'].rename('sim_depth_iot_table_m')

# 時間軸對齊：以 index 合併三條序列
aligned = pd.concat([sim_series, sim_iot_series, obs_series], axis=1).sort_index()
print('station_id:', station_id)
print('mesh2d_face_index:', face_idx)
print('simulation_source:', sim_mode)
aligned.head()


## Step 6. 計算簡單驗證指標
- `MAE`：平均絕對誤差
- `Correlation`：模擬與觀測的一致性

In [ ]:
valid = aligned.dropna(subset=['sim_depth_model_m', 'obs_depth_iot_m'])
mae = (valid['sim_depth_model_m'] - valid['obs_depth_iot_m']).abs().mean()
corr = valid['sim_depth_model_m'].corr(valid['obs_depth_iot_m'])
print(f'Samples: {len(valid)}')
print(f'MAE (m): {mae:.4f}')
print(f'Correlation: {corr:.4f}')


## Step 7. 視覺化比較
圖上三條線分別是：
- `Simulation (model source)`：模型來源（Zarr 或 CSV fallback）
- `Simulation in IoT Parquet`：IoT 表內記錄的模擬欄位
- `Observed IoT`：觀測值

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(aligned.index, aligned['sim_depth_model_m'], label='Simulation (model source)', linewidth=2)
plt.plot(aligned.index, aligned['sim_depth_iot_table_m'], label='Simulation in IoT Parquet', linestyle='--', alpha=0.8)
plt.plot(aligned.index, aligned['obs_depth_iot_m'], label='Observed IoT', linewidth=2)
plt.title(f'Flood Event 20210604 | {station_id} | face={face_idx}')
plt.xlabel('Time (UTC)')
plt.ylabel('Water Depth (m)')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()
